In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="inesc-id/FalAR", 
    repo_type="dataset", local_dir="./FalAR", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1131 files: 100%|██████████| 1131/1131 [00:00<00:00, 2621.41it/s]


'/home/ubuntu/FalAR'

In [3]:
files = glob('FalAR/*/*.parquet')
len(files)

1131

In [4]:
# df = pd.read_parquet(files[0])
# df

In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['wav'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{int(df['speaker_id'].iloc[i])}"
            })
        
    return data

In [6]:
# data = loop((files[:1], 0))

In [ ]:
data = multiprocessing(files, loop, cores = 20)

  2%|▏         | 1/56 [01:48<1:39:12, 108.23s/it]

In [8]:
len(data)

792464

In [9]:
data[0]

{'audio_filename': 'FalAR_audio/FalAR-data-train_3-00024-of-00071_0.mp3',
 'text': 'e opusemos a essa medida e apresentámos este lei tendo fundamentalmente em atenção as questões de princípio invocadas no relatório da comissão nacional de proteção de dados sobre esta matéria que o governo pura e simplesmente ignorou',
 'speaker': 'FalAR_audio_107'}

In [10]:
with open('FalAR.json', 'w') as fopen:
    json.dump(data, fopen)

In [11]:
audio_files = [d['audio_filename'] for d in data]

with open('FalAR-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [12]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'FalAR_audio/FalAR-data-train_3-00024-of-00071_0.mp3',
 'text': 'e opusemos a essa medida e apresentámos este lei tendo fundamentalmente em atenção as questões de princípio invocadas no relatório da comissão nacional de proteção de dados sobre esta matéria que o governo pura e simplesmente ignorou',
 'speaker': 'FalAR_audio_107'}

In [13]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'FalAR')

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00,  5.50ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  80%|███████▉  |  115MB /  144MB,  287MB/s  
Processing Files (0 / 1): 100%|█████████▉|  144MB /  144MB,  239MB/s  
Processing Files (1 / 1): 100%|██████████|  144MB /  144MB,  120MB/s  
Processing Files (1 / 1): 100%|██████████|  144MB /  144MB,  103MB/s  
New Data Upload: 100%|██████████|  144MB /  144MB,  103MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.56s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/620f3db2802913332a01e8f22c4adfaefc3da050', commit_message='Upload dataset', commit_description='', oid='620f3db2802913332a01e8f22c4adfaefc3da050', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [14]:
!zip -rq FalAR_audio_neucodec.zip FalAR_audio_neucodec

In [15]:
!hf upload malaysia-ai/Multilingual-TTS FalAR_audio_neucodec.zip --repo-type=dataset

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  FalAR_audio_neucodec.zip    :   0%|              | 8.47MB / 2.72GB            

Processing Files (0 / 1)      :   0%|              | 8.47MB / 2.72GB, 1.14MB/s  
New Data Upload               :  13%|█▊            | 8.47MB / 67.1MB, 1.14MB/s  

Processing Files (0 / 1)      :   5%|▋             |  133MB / 2.72GB, 17.5MB/s  
New Data Upload               :  66%|█████████▎    |  133MB /  201MB, 17.5MB/s  

Processing Files (0 / 1)      :   9%|█▎            |  244MB / 2.72GB, 31.3MB/s  
New Data Upload               :  91%|████████████▊ |  244MB /  268MB, 31.3MB/s  

Processing Files (0 / 1)      :  13%|█▊            |  355MB / 2.72GB, 44.4MB/s  
New Data Upload               :  88%|████████████▎ |  355MB /  402MB, 44.4MB/s  

Processing Files (0 / 1)      :  17%|██▍           |  476MB / 2.72GB, 58.1MB/s  
New Data Upload       

In [ ]:
!du -hs FalAR_audio